# M2.0b Primer: From Poisson's Equation to a Linear System

Book pointer: chapter 6, sections 6.1 (Laplace's and Poisson's equations), 6.2 (boundary conditions), 6.3 (finite difference methods), 6.4 (implicit methods).

Second of two primers for module 2. The notebooks that follow call relaxation and sparse solvers as finished tools. This one builds them from nothing and explains the mathematics they rest on. It starts with the equation itself and the question of what extra information pins down a solution, moves to one dimension with five grid points where every equation can be written out and every iteration done with a pencil, then to two dimensions, where the only change is that each point has four neighbours instead of two. By the end you will have written the matrix, done Jacobi sweeps by hand, understood why and how fast they converge, seen why the answer is unique, and turned a solved potential back into fields and charges. Section 2 teaches the ideas in prose and can be read on its own; sections 3 and 4 work the numbers.

## 1. Warm-up

Try on paper first. Answers are in section 7.

1. Combine $\mathbf{E} = -\nabla V$ and $\nabla\cdot\mathbf{E} = \rho/\epsilon_0$ into one equation for $V$.
2. What is a boundary value problem? What extra information does a differential equation need to have one solution?
3. Write the finite-difference approximation to $d^2V/dx^2$ at grid point $i$. What is the size of its error?
4. On a grid, what does "$V$ satisfies Laplace's equation" say about $V_i$ and its neighbours?
5. Why does the solution of Laplace's equation with fixed boundary values have no interior maximum?
6. Jacobi on a grid of $N$ points along each side needs about how many sweeps, and why?
7. Once $V$ is known on a grid, how do you get $\mathbf{E}$, and how do you get the charge on a conductor?


## 2. The ideas

**Why solve for $V$ at all.** Module 1 computed $\mathbf{E}$ by adding up Coulomb fields of known charges. That works when you know where every charge is. In most real problems you do not: a metal plate held at 5 volts carries whatever charge it needs to sit at 5 volts, and that charge rearranges itself when another conductor is brought near. What you know is the potential on the conductors, not the charge on them. So the natural unknown is the potential $V(\mathbf{r})$ everywhere in the space between the conductors. It is one scalar function instead of three components, and the field follows from it by one gradient.

**The equation.** Gauss' law in differential form, $\nabla\cdot\mathbf{E} = \rho/\epsilon_0$, combined with $\mathbf{E} = -\nabla V$, gives $\nabla^2V = -\rho/\epsilon_0$. This is Poisson's equation; where $\rho = 0$ it is Laplace's equation, $\nabla^2V = 0$. Primer 00c explained what the Laplacian measures: how far a point sits below the average of its neighbours. So Laplace's equation says that in empty space the potential at every point equals the average of the potential around it. Nothing bulges up or dips down on its own; the only features of $V$ are the ones the boundaries impose. Poisson's equation says the same except that charge is allowed to push a point above or below its neighbours' average, positive charge making a local hill and negative charge a local hollow.

**Why an equation alone is not enough.** $\nabla^2V = 0$ is satisfied by $V = 0$, by $V = 3x$, by $V = x^2 - y^2$, and by infinitely many other functions. A differential equation constrains how $V$ changes from point to point but says nothing about its overall level or tilt. What fixes those is information about $V$ on the edge of the region you are solving in. Give the value of $V$ on the whole boundary (a Dirichlet condition) or the normal derivative $\partial V/\partial n$ on it (a Neumann condition), and there is exactly one solution inside. The equation plus the region plus the boundary data is a boundary value problem. Physically the boundary data comes from three places: conductors, whose surfaces sit at a known potential; symmetry planes, across which the field has no normal component so $\partial V/\partial n = 0$; and distant boundaries, where $V$ is taken as zero because the charges are far away. The last one is an approximation, and how far "far" must be is something you measure rather than assume.

**Discretisation.** A computer cannot hold a function, only numbers at grid points. Replace the second derivative by the difference of neighbouring values, $(V_{i-1} - 2V_i + V_{i+1})/h^2$, and Poisson's equation at a grid point becomes one ordinary linear equation connecting that point to its neighbours. Do this at every interior point and you have as many linear equations as unknowns, with the known boundary values appearing as constants. That is a square linear system $A\mathbf{V} = \mathbf{b}$, and the whole of this primer is about building and solving it. The matrix $A$ is very sparse: each row has three non-zeros in one dimension and five in two, because each point talks only to its immediate neighbours.

**Two ways to solve it.** The first is iteration, called relaxation. Rearrange each equation so that it reads "this point equals the average of its neighbours, plus a source term", start from any guess, and keep replacing every value by what its own equation demands. Each sweep moves information one grid cell, so the number of sweeps grows with the size of the grid, and the longest, smoothest errors die out slowest. Jacobi, Gauss-Seidel and successive over-relaxation are three versions of this, each faster than the last. The second way is to hand the whole sparse matrix to a direct solver, which factorises it once and returns the exact solution of the discrete system in a fraction of a second for grids with hundreds of thousands of points. Relaxation is worth learning because it shows what the equation means; the sparse solve is what you reach for in practice.

**Why both give the same answer.** The discrete problem, like the continuous one, has exactly one solution. The argument is short and is the single most useful theorem in electrostatics: if two solutions share the same boundary values, their difference satisfies Laplace's equation with zero on the boundary, and a function that equals the average of its neighbours cannot have an interior maximum or minimum, so it is zero everywhere. Consequences: any method that satisfies the equation and the boundary conditions has found the solution, however it got there; a clever guess (the method of images, in notebook 03) is as valid as a million sweeps; and a conductor's cavity with no charge inside is field-free.

**Back to physics.** Once $V$ is known on the grid, $\mathbf{E} = -\nabla V$ by central differences gives the field. At a conductor surface the normal field gives the surface charge through $\sigma = \epsilon_0E_n$, and the flux of $\mathbf{E}$ through any box around a conductor gives its total charge. Notebook 02 builds exactly these solvers with Numba and sparse matrices, notebook 03 uses them for conductors and image charges, and notebook 04 turns the field into stored energy and capacitance.


## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sps
import scipy.sparse.linalg as spla
import time
from emlib import fields
from emlib.fields import EPS0

plt.rcParams["figure.figsize"] = (6, 5)
np.set_printoptions(precision=4, suppress=True, linewidth=120)

### 3.1 From Gauss to Poisson, and what a boundary value problem is

Substitute $\mathbf{E} = -\nabla V$ into $\nabla\cdot\mathbf{E} = \rho/\epsilon_0$:
$$\nabla\cdot(-\nabla V) = \rho/\epsilon_0 \quad\Longrightarrow\quad \nabla^2 V = -\frac{\rho}{\epsilon_0}.$$
This is Poisson's equation. Where there is no charge it becomes Laplace's equation, $\nabla^2V = 0$. Both are second-order partial differential equations, and a second-order equation on its own does not pin down a solution: $V = 0$, $V = x$, and $V = x^2 - y^2$ all satisfy Laplace's equation.

Why two pieces of information are needed. In one dimension $d^2V/dx^2 = 0$ integrates twice and each integration introduces a constant: $V = c_1x + c_2$. Two constants, so two conditions. In two or three dimensions the "constants of integration" become whole functions on the boundary: the equation determines the interior once you specify what happens on every piece of the edge. Leave a piece of the boundary unspecified and the solution there can slide freely. Two kinds of condition appear:

- Dirichlet: the value of $V$ is given on the boundary. Physically, a conductor held at a known potential by a battery or by grounding, or a far-away boundary where $V \to 0$. This is the common case and the only one notebook 02 uses.
- Neumann: the normal derivative $\partial V/\partial n$ is given on the boundary. Physically, the normal field $E_n = -\partial V/\partial n$, which by section 3.7 of the previous primer is the same as specifying the surface charge density. The most common Neumann condition is $\partial V/\partial n = 0$ on a plane of symmetry, where no field line crosses the plane; it lets you solve half the problem (section 3.8).

Either kind on each piece of the boundary gives a well-posed boundary value problem with exactly one solution (section 3.7). Mixed problems, Dirichlet on some pieces and Neumann on others, are also fine. The job of a solver is to find that one solution.

A pitfall with far boundaries. Many problems, such as a charged conductor alone in space, have no natural outer boundary. The grid must end somewhere, and setting $V = 0$ on an artificial box pretends that a grounded conductor surrounds the system. The error this makes falls off as the box grows, roughly as the ratio of the system size to the box size for a net-charged system, and faster for a neutral one. The explore task in notebook 02 measures it. Until you have measured it, treat the outer box as part of the physics, not as "infinity".

**Worked example 1.** In one dimension between two parallel plates at $x = 0$ and $x = L$ held at $V = 0$ and $V = V_0$, with no charge in between: $d^2V/dx^2 = 0$, so $V = c_1x + c_2$. The boundary values give $c_2 = 0$ and $c_1 = V_0/L$: $V = V_0x/L$, and $E = -dV/dx = -V_0/L$, uniform. With a uniform charge density $\rho_0$ between the plates: $d^2V/dx^2 = -\rho_0/\epsilon_0$, so $V = -\rho_0x^2/2\epsilon_0 + c_1x + c_2$. With $V(0) = 0$ and $V(L) = V_0$: $c_2 = 0$ and $c_1 = V_0/L + \rho_0L/2\epsilon_0$. The potential bulges upward by a parabola on top of the straight line. These two exact solutions are the test cases for everything below.

**Worked example 2 (concentric spheres).** Two conducting spherical shells, radius $a$ at $V_0$ and radius $b > a$ at $0$, nothing in between. By symmetry $V$ depends on $r$ only, and from primer 00d the radial part of the Laplacian in spherical coordinates is $\frac1{r^2}\frac{d}{dr}\big(r^2\frac{dV}{dr}\big)$. Setting it to zero: $r^2\,dV/dr$ is a constant, call it $-A$, so $dV/dr = -A/r^2$ and $V = A/r + B$. The conditions $V(a) = V_0$ and $V(b) = 0$ give $A/a + B = V_0$ and $A/b + B = 0$, so $A = V_0\,ab/(b - a)$ and $B = -V_0\,a/(b - a)$. With $a = 0.5$, $b = 1$, $V_0 = 1$: $A = 1$, $B = -1$, and $V = 1/r - 1$. Check: $V(0.5) = 2 - 1 = 1$, $V(1) = 0$. The field is $E_r = -dV/dr = A/r^2$, the point-charge form, which is what Gauss' law in primer 00a gave for a spherical capacitor. The charge on the inner shell follows from $E_r = Q/4\pi\epsilon_0r^2$: $Q = 4\pi\epsilon_0A$. So solving Laplace's equation between the shells recovers the Gauss result and additionally tells you how much charge sits on each shell for a given voltage, which Gauss alone did not.

**Worked example 3 (coaxial cylinders).** Inner radius $a$ at $V_0$, outer radius $b$ at $0$, long in $z$. The radial Laplacian in cylindrical coordinates is $\frac1r\frac{d}{dr}\big(r\frac{dV}{dr}\big)$. Setting it to zero: $r\,dV/dr = A$, so $V = A\ln r + B$. The conditions give $A\ln a + B = V_0$ and $A\ln b + B = 0$, hence $A = -V_0/\ln(b/a)$ and $B = -A\ln b$. With $a = 0.5$, $b = 1$, $V_0 = 1$: $A = -1/\ln2 = -1.4427$, $B = 0$, $V = -\ln r/\ln 2$. Check: $V(0.5) = 1$, $V(1) = 0$. The field is $E_r = -A/r = 1.4427/r$, the line-charge form $\lambda/2\pi\epsilon_0r$ from primer 00a, with $\lambda = -2\pi\epsilon_0A$. The code evaluates both closed forms on a radial grid, confirms the boundary values, and confirms that the radial Laplacian of each is zero to rounding error. They return in notebook 04 as the two capacitors with closed-form answers.


In [ ]:
a, b, V0 = 0.5, 1.0, 1.0
r = np.linspace(a, b, 2001); h_r = r[1] - r[0]

A_s, B_s = V0 * a * b / (b - a), -V0 * a / (b - a)
V_sph = A_s / r + B_s
lap_sph = np.gradient(r**2 * np.gradient(V_sph, h_r), h_r) / r**2
print(f"spheres:   A = {A_s:.4f}, B = {B_s:.4f};  V(a) = {V_sph[0]:.4f}, V(b) = {V_sph[-1]:.4f};  max |radial Laplacian| (interior) = {np.abs(lap_sph[5:-5]).max():.1e}")

A_c = -V0 / np.log(b / a); B_c = -A_c * np.log(b)
V_cyl = A_c * np.log(r) + B_c
lap_cyl = np.gradient(r * np.gradient(V_cyl, h_r), h_r) / r
print(f"cylinders: A = {A_c:.4f}, B = {B_c:.4f};  V(a) = {V_cyl[0]:.4f}, V(b) = {V_cyl[-1]:.4f};  max |radial Laplacian| (interior) = {np.abs(lap_cyl[5:-5]).max():.1e}")

plt.plot(r, V_sph, label="spheres: 1/r - 1"); plt.plot(r, V_cyl, label="cylinders: -ln r / ln 2"); plt.plot(r, (b - r) / (b - a), "k:", label="plates: straight line")
plt.xlabel("r"); plt.ylabel("V"); plt.legend(); plt.title("Laplace between two conductors at 1 V and 0 V"); plt.show()

Read the plot. All three potentials fall from 1 to 0 across the same gap, but the curved geometries fall faster near the inner conductor, where the field lines are crowded together. That crowding is the $1/r$ and $1/r^2$ of the field, and it is what a flat-plate approximation misses.

### 3.2 One dimension on five points, by hand

Where the three-point formula comes from. Expand $V$ in a Taylor series about $x_i$ for a step of $\pm h$:
$$V_{i\pm1} = V_i \pm hV_i' + \tfrac12h^2V_i'' \pm \tfrac16h^3V_i''' + \tfrac1{24}h^4V_i'''' + \dots$$
Add the two expansions: the odd powers cancel, leaving $V_{i+1} + V_{i-1} = 2V_i + h^2V_i'' + \tfrac1{12}h^4V_i'''' + \dots$. Solve for the second derivative:
$$V_i'' = \frac{V_{i-1} - 2V_i + V_{i+1}}{h^2} - \frac{h^2}{12}V_i'''' + \dots$$
The leading error is proportional to $h^2$ times the fourth derivative. Halve $h$ and the error drops by four. And if $V$ is a polynomial of degree three or less, its fourth derivative is zero and the formula is exact. That is why the straight line and the parabola of worked example 1 come out exactly on any grid, and why a sine wave does not.

Now take $L = 1$, $V(0) = 0$, $V(1) = 1$, and a grid of five points $x_i = 0, 0.25, 0.5, 0.75, 1$, so $h = 0.25$. The values $V_0 = 0$ and $V_4 = 1$ are known; $V_1, V_2, V_3$ are the unknowns. Poisson's equation at interior point $i$ becomes the linear equation
$$V_{i-1} - 2V_i + V_{i+1} = -\frac{h^2\rho_i}{\epsilon_0}.$$

**Worked example 4 (Laplace, $\rho = 0$).** The three equations:
$$i = 1:\ V_0 - 2V_1 + V_2 = 0 \ \Rightarrow\ -2V_1 + V_2 = 0,$$
$$i = 2:\ V_1 - 2V_2 + V_3 = 0,$$
$$i = 3:\ V_2 - 2V_3 + V_4 = 0 \ \Rightarrow\ V_2 - 2V_3 = -1.$$
Known boundary values have moved to the right-hand side. In matrix form:
$$\begin{pmatrix} -2 & 1 & 0 \\ 1 & -2 & 1 \\ 0 & 1 & -2 \end{pmatrix}\begin{pmatrix} V_1 \\ V_2 \\ V_3 \end{pmatrix} = \begin{pmatrix} 0 \\ 0 \\ -1 \end{pmatrix}.$$
Solve by substitution: from the first, $V_2 = 2V_1$. The second gives $V_1 - 4V_1 + V_3 = 0$, so $V_3 = 3V_1$. The third: $2V_1 - 6V_1 = -1$, so $V_1 = 0.25$, then $V_2 = 0.5$, $V_3 = 0.75$. Exactly $V = x$, as the Taylor argument predicted.

**Worked example 5 (Poisson, $\rho_0/\epsilon_0 = 8$).** The right-hand side gains $-h^2\rho/\epsilon_0 = -0.0625\cdot8 = -0.5$ at every interior point: $\mathbf{b} = (-0.5, -0.5, -1.5)$. The exact solution from worked example 1 is $V = -4x^2 + 5x$, giving $V(0.25) = 1.0$, $V(0.5) = 1.5$, $V(0.75) = 1.5$. Substitution again: first row $V_2 = 2V_1 - 0.5$; second row $V_1 - 2(2V_1 - 0.5) + V_3 = -0.5$ gives $V_3 = 3V_1 - 1.5$; third row $(2V_1 - 0.5) - 2(3V_1 - 1.5) = -1.5$ gives $-4V_1 = -4$, $V_1 = 1$, then $V_2 = 1.5$, $V_3 = 1.5$. Exact again, because the solution is a quadratic.


In [ ]:
A = np.array([[-2.0, 1, 0], [1, -2, 1], [0, 1, -2]])
b_laplace = np.array([0.0, 0, -1])
b_poisson = np.array([-0.5, -0.5, -1.5])
print("Laplace, three unknowns:", np.linalg.solve(A, b_laplace), "   exact V = x at 0.25, 0.5, 0.75: [0.25 0.5 0.75]")
print("Poisson, rho/eps0 = 8:  ", np.linalg.solve(A, b_poisson), "   exact V = -4x^2 + 5x:           [1.   1.5  1.5 ]")

### 3.3 The same thing for any number of points

With $N$ interior points the matrix is $N\times N$ with $-2$ on the diagonal and $1$ on both neighbouring diagonals (tridiagonal). The right-hand side is $-h^2\rho_i/\epsilon_0$, with the two boundary values subtracted from the first and last entries. Build it, solve it, and compare to the exact parabola as the grid is refined. Because the solution is a quadratic the error is at rounding level for every $N$; for a smooth non-polynomial solution it falls as $h^2$, which the exercise at the end measures.

One thing to know about this matrix: it becomes harder to solve accurately as $N$ grows. Its eigenvalues range from about $-4$ down to about $-\pi^2h^2$, so the ratio of largest to smallest (the condition number) grows like $N^2$. For a direct solver that only means a few more digits lost to rounding, harmless at the sizes used here. For iterative methods it is the reason the sweep count grows with $N^2$, as section 3.4 shows.


In [ ]:
def poisson_1d_matrix(N, h, rho_over_eps0, V_left, V_right):
    """Tridiagonal system for N interior points. Returns A (N x N) and b (N,)."""
    A = -2 * np.eye(N) + np.eye(N, k=1) + np.eye(N, k=-1)
    b = -h**2 * rho_over_eps0 * np.ones(N)
    b[0] -= V_left; b[-1] -= V_right
    return A, b

for N in (3, 7, 15, 63):
    h = 1.0 / (N + 1)
    xi = h * np.arange(1, N + 1)
    A, b = poisson_1d_matrix(N, h, 8.0, 0.0, 1.0)
    V = np.linalg.solve(A, b)
    print(f"N = {N:3d} interior points, h = {h:.4f}:  max |V - exact| = {np.abs(V - (-4 * xi**2 + 5 * xi)).max():.1e}")
print("\nthe 7-point matrix:"); print(poisson_1d_matrix(7, 0.125, 8.0, 0.0, 1.0)[0])

### 3.4 Relaxation: solving by averaging

Rearrange the equation at point $i$ to isolate $V_i$:
$$V_i = \frac{V_{i-1} + V_{i+1}}{2} + \frac{h^2\rho_i}{2\epsilon_0}.$$
Each point should equal the average of its neighbours, plus a source term. Jacobi's method takes that literally: start from any guess, compute a new value at every point from the old neighbours, repeat. Gauss-Seidel does the same but uses each new value as soon as it exists, sweeping left to right, which roughly halves the number of sweeps. Both stop when the largest change in a sweep drops below a tolerance. Neither needs the matrix to be written down: the update rule is the matrix.

**Worked example 6 (three Jacobi sweeps by hand).** Laplace on the five-point grid, start from $V = (0, 0, 0, 0, 1)$ (boundary values in place, interior zero).

Sweep 1, every interior point from the old neighbours: $V_1 = (0 + 0)/2 = 0$, $V_2 = (0 + 0)/2 = 0$, $V_3 = (0 + 1)/2 = 0.5$. Result $(0, 0, 0, 0.5, 1)$.

Sweep 2: $V_1 = (0 + 0)/2 = 0$, $V_2 = (0 + 0.5)/2 = 0.25$, $V_3 = (0 + 1)/2 = 0.5$. Result $(0, 0, 0.25, 0.5, 1)$.

Sweep 3: $V_1 = 0.125$, $V_2 = 0.25$, $V_3 = 0.625$. The boundary information creeps inward one point per sweep and the values climb toward $(0.25, 0.5, 0.75)$. That creeping is the first reason the number of sweeps grows with the grid size: information has to cross the whole grid.

**Worked example 7 (two Gauss-Seidel sweeps by hand).** Same start. $V_1 = (0 + 0)/2 = 0$. $V_2 = (V_1^{\text{new}} + 0)/2 = 0$. $V_3 = (V_2^{\text{new}} + 1)/2 = 0.5$. Sweep 2: $V_1 = 0$, $V_2 = (0 + 0.5)/2 = 0.25$, $V_3 = (0.25 + 1)/2 = 0.625$. Already ahead of Jacobi at $V_3$, because it used the fresh $V_2$.


In [ ]:
def jacobi_sweep(V, src):
    Vn = V.copy()
    Vn[1:-1] = 0.5 * (V[:-2] + V[2:]) + 0.5 * src[1:-1]
    return Vn, np.abs(Vn - V).max()

def gauss_seidel_sweep(V, src):
    change = 0.0
    for i in range(1, len(V) - 1):
        new = 0.5 * (V[i - 1] + V[i + 1]) + 0.5 * src[i]
        change = max(change, abs(new - V[i])); V[i] = new
    return V, change

V = np.array([0.0, 0, 0, 0, 1.0]); src = np.zeros(5)
print("Jacobi by hand, sweep by sweep:")
for k in range(1, 4):
    V, _ = jacobi_sweep(V, src); print(f"  after sweep {k}: {V}")
V = np.array([0.0, 0, 0, 0, 1.0])
print("Gauss-Seidel:")
for k in range(1, 3):
    V, _ = gauss_seidel_sweep(V, src); print(f"  after sweep {k}: {V}")

# Convergence to tolerance on a larger grid, both methods.
N = 50; src = np.zeros(N + 2)
for name, sweep in (("Jacobi", jacobi_sweep), ("Gauss-Seidel", gauss_seidel_sweep)):
    V = np.zeros(N + 2); V[-1] = 1.0; history = []
    for k in range(100000):
        V, change = sweep(V, src); history.append(change)
        if change < 1e-8: break
    plt.semilogy(history, label=f"{name}: {len(history)} sweeps")
plt.xlabel("sweep"); plt.ylabel("largest change"); plt.legend(); plt.title("relaxation on 50 interior points"); plt.show()

**Why averaging converges, and how fast.** Write $V = V_{\text{exact}} + e$. The exact solution satisfies the averaging equation, so the error $e$ satisfies the pure Laplace version: each sweep replaces $e_i$ by the average of its neighbours, with $e = 0$ on the boundary. Averaging never increases the largest value and smooths every bump, and the only function that equals the average of its neighbours everywhere and vanishes on the boundary is zero (that is the uniqueness argument of section 3.7). So $e \to 0$.

How fast is a sharper question, and the answer explains everything about relaxation. Any error on $N$ interior points with $h = 1/(N + 1)$ can be written as a sum of sine modes $e_i = \sum_k c_k\sin(k\pi x_i)$, $k = 1, \dots, N$, because those modes vanish at both ends. Apply one Jacobi sweep to a single mode: using $\sin(k\pi(x_i \pm h)) = \sin(k\pi x_i)\cos(k\pi h) \pm \cos(k\pi x_i)\sin(k\pi h)$,
$$\frac{\sin(k\pi x_{i-1}) + \sin(k\pi x_{i+1})}{2} = \cos(k\pi h)\,\sin(k\pi x_i).$$
So a Jacobi sweep leaves each mode's shape alone and multiplies its amplitude by $\cos(k\pi h)$. Short, wiggly modes (large $k$) have $\cos(k\pi h)$ well below one and die in a few sweeps. The longest mode, $k = 1$, has $\cos(\pi h) \approx 1 - \pi^2h^2/2$, barely less than one. It needs about $2/(\pi^2h^2) \approx 2N^2/\pi^2$ sweeps to shrink by a factor $e$, which is where the $N^2$ scaling comes from. Gauss-Seidel squares the factor to $\cos^2(\pi h)$, halving the count.

**Worked example 8.** $N = 50$ interior points, $h = 1/51$. Jacobi factor for the slowest mode: $\cos(\pi/51) = 0.99810$. To reduce the error by $10^{-8}$: $\ln(10^{-8})/\ln(0.99810) = -18.42/(-0.001899) \approx 9700$ sweeps. Compare that with the Jacobi count in the plot above (started from a step, whose error is mostly in the slowest mode, and stopped on the per-sweep change rather than the error, which brings the count down somewhat). Gauss-Seidel: factor $0.99810^2 = 0.99621$, about $4850$ sweeps. The code measures the factor directly by applying one sweep to pure modes.

**Over-relaxation.** Gauss-Seidel moves each value toward its neighbour average. If the error is a slow mode, the move is in the right direction but too short, so push further: $V_i \leftarrow V_i + \omega(V_i^{\text{GS}} - V_i)$ with $\omega$ between 1 and 2. This is successive over-relaxation (SOR). For the right $\omega$ the slowest mode's factor drops from $1 - \pi^2h^2$ to about $1 - 2\pi h$, and the sweep count from order $N^2$ to order $N$. The near-optimal value used by `emlib.relax.sor_omega` is
$$\omega_{\text{opt}} = \frac{2}{1 + \sqrt{1 - \rho_J^2}}, \qquad \rho_J = \cos(\pi h),$$
where $\rho_J$ is the Jacobi factor of the slowest mode (on a rectangular grid, the average of the two directions' cosines). With $\rho_J$ close to one the square root is small and $\omega_{\text{opt}}$ is close to two: for $N = 50$, $\rho_J = 0.99810$, $\sqrt{1 - \rho_J^2} = 0.0616$, $\omega_{\text{opt}} = 1.884$. Push too far ($\omega \ge 2$) and the iteration diverges. The explore task in section 5 measures the gain.


In [ ]:
N = 50; h = 1.0 / (N + 1); x_nodes = h * np.arange(N + 2)
print("Jacobi damping factor per sweep, measured on a pure sine mode vs cos(k pi h):")
for k in (1, 2, 5, 25, 50):
    e = np.sin(k * np.pi * x_nodes); e[0] = e[-1] = 0.0
    e_new, _ = jacobi_sweep(e, np.zeros_like(e))
    measured = e_new[N // 2] / e[N // 2] if abs(e[N // 2]) > 1e-12 else e_new[N // 2 + 1] / e[N // 2 + 1]
    print(f"  k = {k:2d}: measured {measured:+.5f}   cos(k pi h) = {np.cos(k * np.pi * h):+.5f}")
rho_J = np.cos(np.pi * h)
print(f"\nslowest mode: factor {rho_J:.5f};  sweeps for 1e-8 with Jacobi ~ {np.log(1e-8) / np.log(rho_J):.0f}, Gauss-Seidel ~ {np.log(1e-8) / np.log(rho_J**2):.0f}")
print(f"SOR omega_opt = 2 / (1 + sqrt(1 - rho_J^2)) = {2 / (1 + np.sqrt(1 - rho_J**2)):.4f}")

### 3.5 Two dimensions

Nothing new happens except that each point has four neighbours. The five-point stencil (primer 00c) gives
$$V_{i-1,j} + V_{i+1,j} + V_{i,j-1} + V_{i,j+1} - 4V_{i,j} = -\frac{h^2\rho_{i,j}}{\epsilon_0},$$
one equation per interior node, and the relaxation update is "a quarter of the sum of the four neighbours plus $h^2\rho/4\epsilon_0$". The Taylor argument of section 3.2 applies along each axis separately, so the error is again of order $h^2$.

To write a matrix we need to number the unknowns. A grid point has two indices, a matrix row has one, so pick a rule: row by row, $k = i\cdot n_y + j$, with $n_y$ the number of grid points along $y$. Then the neighbour $(i, j \pm 1)$ sits at $k \pm 1$ and the neighbour $(i \pm 1, j)$ at $k \pm n_y$. The matrix has $-4$ on the diagonal and four off-diagonals of $1$, at distances $1$ and $n_y$. Any consistent numbering works; this one makes the structure easy to see. The cell prints it for a $3\times3$ block of unknowns.

**Worked example 9.** A $3\times3$ interior ($n = 5$ points per side, $h = 0.25$), boundary at $V = 0$ except the top edge ($j = 4$) at $V = 1$. Number the nine unknowns $(1,1), (1,2), (1,3), (2,1), \dots, (3,3)$ as $k = 0, \dots, 8$. Node $(2, 3)$, the middle of the top interior row, is $k = 5$. Its neighbours are $(1, 3)$ at $k = 2$, $(3, 3)$ at $k = 8$, $(2, 2)$ at $k = 4$, and the boundary node $(2, 4) = 1$. Its equation: $V_{1,3} + V_{3,3} + V_{2,2} - 4V_{2,3} = -1$. So row 5 of $A$ has $-4$ in column 5 and $1$ in columns 2, 4 and 8, and $b_5 = -1$: the known $1$ went to the right-hand side with a minus sign. The centre node $(2, 2)$, $k = 4$, has four unknown neighbours ($k = 1, 7, 3, 5$) and right-hand side $0$. By symmetry the solution has $V_{1,j} = V_{3,j}$, and the code confirms the whole $3\times3$ answer.


In [ ]:
def poisson_2d_dense(n, V_boundary, rho_over_eps0=None):
    """Dense A V = b for an n x n grid with the outer ring fixed to V_boundary. Returns A, b, and an index map."""
    h = 1.0 / (n - 1)
    interior = [(i, j) for i in range(1, n - 1) for j in range(1, n - 1)]
    index = {ij: k for k, ij in enumerate(interior)}
    m = len(interior)
    A = np.zeros((m, m)); b = np.zeros(m)
    for (i, j), k in index.items():
        A[k, k] = -4.0
        if rho_over_eps0 is not None: b[k] = -h**2 * rho_over_eps0[i, j]
        for ni, nj in ((i - 1, j), (i + 1, j), (i, j - 1), (i, j + 1)):
            if (ni, nj) in index: A[k, index[(ni, nj)]] = 1.0
            else: b[k] -= V_boundary[ni, nj]
    return A, b, index

n = 5
Vb = np.zeros((n, n)); Vb[:, -1] = 1.0                      # top edge (j = n-1) at 1 V
A, b, index = poisson_2d_dense(n, Vb)
print("A for the 3x3 interior (unknowns numbered row by row):"); print(A)
print("b:", b)
sol = np.linalg.solve(A, b)
V = Vb.copy()
for (i, j), k in index.items(): V[i, j] = sol[k]
print("\nV on the 5x5 grid (rows are i = x index, columns are j = y index):"); print(V)

### 3.6 Fixed nodes with a boolean mask, and the sparse matrix

Two practical tools that notebook 02 uses without comment.

Masks. Conductors inside the region are nodes whose value is known. The clean way to say "these nodes are fixed" is a boolean array `fixed` of the same shape as `V`, `True` where the value is given. `V[~fixed]` selects the unknowns, `fixed[a:b, a:b] = True` marks a square block as a conductor, and in a relaxation sweep you simply reset the fixed nodes after each update. The outer boundary is always fixed.

Sparse matrices. For an $n\times n$ grid there are about $n^2$ unknowns and the dense matrix has $n^4$ entries. At $n = 200$ that is $1.6\times10^9$ numbers, $13\,$GB, nearly all zeros. Each row has at most five non-zeros. SciPy's sparse formats store only those. The compressed sparse row (CSR) format keeps three flat arrays: the non-zero values, the column index of each, and for each row the position in those arrays where that row starts. Assembling one is a matter of building three lists (row index, column index, value), handing them to `scipy.sparse.csr_matrix`, and solving with `spsolve`. A sparse direct solver factorises $A$ into triangular factors once, exploiting the fact that most entries are zero, then solves by two cheap substitutions; for a 2D grid the factorisation costs roughly $n^3$ operations, so a $200\times200$ grid takes well under a second. The assembly below is the same loop as the dense version, appending to lists instead of writing into a big array; the vectorised version in `emlib.relax.solve_sparse` does the same thing without the Python loop.

The cell solves the same problem with a conductor block inside three ways (Jacobi with a mask, dense, sparse) and shows that the sparse assembly is the only one that scales.


In [ ]:
def build_problem(n):
    V0 = np.zeros((n, n)); fixed = np.zeros((n, n), dtype=bool)
    V0[:, -1] = 1.0
    fixed[0, :] = fixed[-1, :] = fixed[:, 0] = fixed[:, -1] = True
    a, b_ = int(0.35 * n), int(0.55 * n)
    fixed[a:b_, a:b_] = True                                   # a grounded block: V0 is already 0 there
    return V0, fixed

def jacobi_2d(V0, fixed, tol=1e-6, max_sweeps=200000):
    V = V0.copy()
    for k in range(max_sweeps):
        Vn = V.copy()
        Vn[1:-1, 1:-1] = 0.25 * (V[2:, 1:-1] + V[:-2, 1:-1] + V[1:-1, 2:] + V[1:-1, :-2])
        Vn[fixed] = V0[fixed]                                  # fixed nodes keep their values
        change = np.abs(Vn - V).max(); V = Vn
        if change < tol: return V, k + 1
    return V, max_sweeps

def sparse_2d(V0, fixed):
    n = V0.shape[0]; idx = np.arange(n * n).reshape(n, n)
    rows, cols, vals = [], [], []; b = np.zeros(n * n)
    for i in range(n):
        for j in range(n):
            k = idx[i, j]
            if fixed[i, j]:
                rows.append(k); cols.append(k); vals.append(1.0); b[k] = V0[i, j]
            else:
                rows.append(k); cols.append(k); vals.append(-4.0)
                for ni, nj in ((i - 1, j), (i + 1, j), (i, j - 1), (i, j + 1)):
                    rows.append(k); cols.append(idx[ni, nj]); vals.append(1.0)
    A = sps.csr_matrix((vals, (rows, cols)), shape=(n * n, n * n))
    return spla.spsolve(A, b).reshape(n, n)

n = 41
V0, fixed = build_problem(n)
t = time.time(); V_j, sweeps = jacobi_2d(V0, fixed, tol=1e-11); t_j = time.time() - t
t = time.time(); V_s = sparse_2d(V0, fixed); t_s = time.time() - t
print(f"n = {n}: Jacobi {sweeps} sweeps in {t_j:.2f} s, sparse direct in {t_s:.3f} s, max |difference| = {np.abs(V_j - V_s).max():.1e}")
print(f"dense matrix would have {(n * n) ** 2:,} entries; the sparse one stores {5 * (~fixed).sum() + fixed.sum():,}")

x = np.linspace(0, 1, n); X, Y = np.meshgrid(x, x, indexing="ij")
plt.contourf(X, Y, V_s, levels=21, cmap="viridis"); plt.colorbar(label="V")
plt.contour(X, Y, V_s, levels=11, colors="w", linewidths=0.5); plt.gca().set_aspect("equal"); plt.title("top edge at 1 V, grounded box and block"); plt.show()

### 3.7 Uniqueness, and why it matters

Suppose $V_1$ and $V_2$ both satisfy Poisson's equation in a region with the same values on the boundary. Their difference $W = V_1 - V_2$ satisfies Laplace's equation (the sources cancel) and is zero on the boundary. A solution of Laplace's equation equals the average of its neighbours everywhere, so it cannot have a maximum or minimum inside the region: an interior maximum would be larger than its neighbours, which contradicts being their average. So $W$ attains its largest and smallest values on the boundary, where it is zero. Hence $W = 0$ everywhere and $V_1 = V_2$. The same argument works with Neumann conditions, except that $W$ is then determined only up to an additive constant, which is why a pure Neumann problem needs one value pinned somewhere.

Consequences you will use.

- Any method that produces a function satisfying the equation and the boundary conditions has produced the solution. Jacobi, Gauss-Seidel, the sparse solver, a lucky guess: all the same answer. The code checks this by starting Jacobi from a random guess.
- In notebook 03, the method of images replaces a conductor by a fictitious charge. The resulting potential satisfies Laplace's equation outside the conductor and the right boundary values on it, so by uniqueness it is the solution there, however strange the construction looks.
- The field inside a closed conductor with no charge in its cavity is zero: $V$ is constant on the cavity wall, the constant satisfies Laplace's equation with those boundary values, so it is the solution.

**The mean-value property.** The "average of its neighbours" statement has an exact continuous form: for a function satisfying Laplace's equation, the average of $V$ over any circle (in 2D) or sphere (in 3D) equals its value at the centre. Primer 00c checked this numerically for $x^2 - y^2$. It holds for every harmonic function, and the maximum principle above is its immediate consequence: a value cannot exceed the average of values around it.

**Earnshaw's theorem.** A charge $q$ placed in an electrostatic field made by other fixed charges cannot sit in stable equilibrium. Stability would need the potential energy $qV$ to have a minimum at the point, so $V$ would have a local minimum (for $q > 0$) in a charge-free region. But $V$ satisfies Laplace's equation there and has no interior extrema. So the charge can be in equilibrium (where $\mathbf{E} = 0$) but any small push will carry it off in at least one direction. This is why electrostatic traps need moving parts or magnetic fields, and why the equilibrium points you find in potential plots are always saddles.


In [ ]:
rng = np.random.default_rng(0)
V_random_start = V0.copy(); V_random_start[~fixed] = rng.uniform(-5, 5, size=(~fixed).sum())
V_r, sweeps_r = jacobi_2d(V_random_start, fixed, tol=1e-11)
V_z, sweeps_z = jacobi_2d(V0, fixed, tol=1e-11)
print(f"from a random guess in [-5, 5]: {sweeps_r} sweeps; from zero: {sweeps_z} sweeps; max |difference| = {np.abs(V_r - V_z).max():.1e}")
print(f"interior maximum of the Laplace solution: {V_s[~fixed].max():.4f}  <=  boundary maximum {V_s[fixed].max():.4f}   (no interior extremum)")

### 3.8 Neumann conditions and symmetry planes

A given normal derivative is handled with a ghost point. At the left boundary $x_0$, require $dV/dx = g$. Place an imaginary point $x_{-1}$ and use the central difference $(V_1 - V_{-1})/2h = g$, so $V_{-1} = V_1 - 2hg$. Then the Poisson equation at $x_0$ itself becomes an equation for the unknown $V_0$: $V_{-1} - 2V_0 + V_1 = -h^2\rho_0/\epsilon_0$, that is $2V_1 - 2V_0 = -h^2\rho_0/\epsilon_0 + 2hg$. The boundary node has joined the unknowns, with a modified row. Using a central difference for the derivative keeps the whole scheme second order; a one-sided difference would be simpler and would quietly reduce the accuracy to first order.

**Worked example 10.** Laplace on $[0, 1]$ with $V'(0) = 2$ and $V(1) = 1$. The exact solution is the line with slope 2 through $(1, 1)$: $V = 2x - 1$. The code assembles the modified first row and recovers it.

**Worked example 11 (symmetry plane).** The commonest Neumann condition is $g = 0$. Poisson's equation on $[-1, 1]$ with $V(\pm1) = 0$ and uniform $\rho/\epsilon_0 = 2$ has the solution $V = 1 - x^2$. The problem is symmetric about $x = 0$, so the solution is too, so $V'(0) = 0$: no field crosses the midplane. That means the right half alone, $[0, 1]$ with $V'(0) = 0$ and $V(1) = 0$, is a complete boundary value problem with the same solution, and it has half as many unknowns. In two and three dimensions the saving is a factor of two per symmetry plane, and the condition is the same: on a plane of mirror symmetry, $\partial V/\partial n = 0$. The ghost-point row at $x_0 = 0$ is $2V_1 - 2V_0 = -h^2\cdot2$, with the $2hg$ term absent. The code solves the half problem and compares with $1 - x^2$.


In [ ]:
N_total = 9                                                 # nodes 0..8, node 8 fixed at V = 1, node 0 has a Neumann condition
h = 1.0 / (N_total - 1); g = 2.0
m = N_total - 1                                              # unknowns: nodes 0..7
A = -2 * np.eye(m) + np.eye(m, k=1) + np.eye(m, k=-1)
A[0, 1] = 2.0                                                # ghost-point row: 2 V_1 - 2 V_0
b = np.zeros(m); b[0] = 2 * h * g; b[-1] -= 1.0
V = np.linalg.solve(A, b)
xs = h * np.arange(m)
print("worked example 10, solver:", V); print("exact 2x - 1:            ", 2 * xs - 1)

# Worked example 11: the half problem [0, 1] with V'(0) = 0 (symmetry plane) and V(1) = 0, rho/eps0 = 2.
A = -2 * np.eye(m) + np.eye(m, k=1) + np.eye(m, k=-1); A[0, 1] = 2.0
b = np.full(m, -h**2 * 2.0)                                  # Poisson source at every unknown, including the symmetry node
V_half = np.linalg.solve(A, b)
print("\nworked example 11, half domain:", V_half); print("exact 1 - x^2:                ", 1 - xs**2)
print(f"max error {np.abs(V_half - (1 - xs**2)).max():.1e}  (exact to rounding: the solution is a quadratic)")

### 3.9 From $V$ to everything else

A solved potential is only the start. The quantities you actually want, and how each comes out of the grid:

The field. $\mathbf{E} = -\nabla V$ by central differences at interior nodes, $E_x \approx -(V_{i+1,j} - V_{i-1,j})/2h$, which is `np.gradient` with a minus sign (`emlib.fields.gradient` stacks the components). The error is of order $h^2$ away from conductors.

The field at a conductor surface. A central difference at the last free node mixes in the fixed node on the other side, which is fine for the field value there, but the quantity you usually want is the field at the surface itself. Use the one-sided difference between the surface node and its free neighbour: at a conductor face whose outward normal is $+\hat{\mathbf{y}}$, $E_n \approx -(V_{i,j+1} - V_{i,j})/h$ with $(i, j)$ on the surface. This is first-order accurate, and it is what notebook 03 uses.

The surface charge. Primer 00a, section 3.7: just outside a conductor $E_n = \sigma/\epsilon_0$, so $\sigma = \epsilon_0E_n$. Positive where field lines leave the conductor, negative where they arrive. The sign is automatic once $E_n$ is measured along the outward normal.

The total charge on a conductor. Two routes. Add up $\sigma$ over the surface, or apply Gauss' law: the charge equals $\epsilon_0$ times the outward flux of $\mathbf{E}$ through any closed surface around the conductor, which on a grid is `emlib.fields.box_flux` on a box one or two cells outside it. The two agree to the accuracy of the differences involved, and the box route is less sensitive to the staircasing of a non-rectangular conductor.

The energy. The field stores energy with density $\tfrac12\epsilon_0|\mathbf{E}|^2$, and integrating it over the grid gives the total, from which capacitance follows as $C = 2U/V_0^2$. Primer 00c for this module derives it and notebook 04 uses it.

**Worked example 12.** Parallel plates $L = 1\,$m apart at $V_0 = 1\,$V and $0$. From worked example 1, $E = V_0/L = 1\,$V/m pointing from the high plate to the low one. On the high plate the outward normal points into the gap, along $\mathbf{E}$, so $E_n = +1\,$V/m and $\sigma = \epsilon_0\cdot1 = 8.85\times10^{-12}\,$C/m$^2$. On the low plate the outward normal points against $\mathbf{E}$, $E_n = -1$, $\sigma = -8.85\times10^{-12}\,$C/m$^2$. Equal and opposite, as a capacitor must be. Per unit depth, a plate of width $W = 1$ carries $Q = \sigma W = 8.85\times10^{-12}\,$C/m.

The code takes the sparse solution of section 3.6 (grounded box and block, top edge at 1 V) and computes the field, the surface charge on each face of the block by one-sided differences, and the total induced charge both by summing $\sigma$ and by a Gaussian box one cell outside the block. The block sits in a field that points downward from the 1 V edge toward it, so its upper face collects negative charge and its lower face, facing the grounded bottom edge, carries much less.


In [ ]:
h = 1.0 / (n - 1)
E = -fields.gradient(V_s, h, h)                             # (n, n, 2): E_x, E_y at every node
a_, b_ = int(0.35 * n), int(0.55 * n)                        # the block occupies nodes a_ .. b_-1 along both axes

# One-sided normal field on each face of the block, then sigma = eps0 E_n. Outward normals: +x, -x, +y, -y.
faces = {
    "+x face (i = b_-1)": -(V_s[b_, a_:b_] - V_s[b_ - 1, a_:b_]) / h,
    "-x face (i = a_)":   -(V_s[a_ - 1, a_:b_] - V_s[a_, a_:b_]) / h,
    "+y face (j = b_-1)": -(V_s[a_:b_, b_] - V_s[a_:b_, b_ - 1]) / h,
    "-y face (j = a_)":   -(V_s[a_:b_, a_ - 1] - V_s[a_:b_, a_]) / h,
}
Q_sigma = 0.0
print("surface charge on the grounded block (per unit depth), from sigma = eps0 E_n:")
for name, En in faces.items():
    q_face = EPS0 * np.sum(En) * h
    Q_sigma += q_face
    print(f"  {name:20s}: mean sigma = {EPS0 * En.mean():+.3e} C/m^2   charge = {q_face:+.3e} C/m")
Q_box = EPS0 * fields.box_flux(E, (h, h), (a_ - 1, a_ - 1), (b_, b_))      # box one cell outside the block
print(f"\ntotal induced charge: sum of sigma = {Q_sigma:+.4e} C/m    Gauss box = {Q_box:+.4e} C/m    ratio {Q_sigma / Q_box:.3f}")

# Worked example 12 on a grid: parallel plates, top edge at 1 V and bottom at 0 V, side edges fixed to the exact linear V.
Vp = np.zeros((n, n)); fp = np.zeros((n, n), dtype=bool)
yy = np.linspace(0, 1, n); Vp[:, :] = yy[None, :]; fp[:, 0] = fp[:, -1] = fp[0, :] = fp[-1, :] = True
Ep = -fields.gradient(Vp, h, h)
sigma_top = EPS0 * (-(Vp[n // 2, -2] - Vp[n // 2, -1]) / h)            # outward normal of the top plate points down (-y): E_n = -E_y
print(f"parallel plates: E_y = {Ep[n // 2, n // 2, 1]:+.3f} V/m, sigma on the 1 V plate = {sigma_top:+.3e} C/m^2  (hand: +{EPS0:.3e})")

A note on the ratio printed above. The sum of $\sigma$ from one-sided differences and the flux through the box one cell out agree to the printed digits. That is not luck: the five-point equations satisfied at every free node are a discrete form of Gauss' law, so the net flux through any closed loop of grid faces enclosing only the block is the same whichever loop you take, and the one-sided surface sum is one such loop. Both are still only first-order approximations to the continuous charge, and both improve with a finer grid. The box route is the one to prefer for a staircased, non-rectangular conductor, where one-sided differences along a jagged surface are ill-defined; notebook 03 uses it.

### Exercise: assemble the 1D system yourself

Replace the body of `my_poisson_1d` so that it builds and solves the tridiagonal system for $N$ interior points with Dirichlet values at both ends and a given $\rho/\epsilon_0$ array. The check uses a non-uniform source $\rho/\epsilon_0 = \pi^2\sin(\pi x)$ with $V(0) = V(1) = 0$, whose exact solution is $V = \sin(\pi x)$, and requires second-order convergence: the error must fall by about four each time $h$ halves, as the Taylor argument of section 3.2 predicts for a solution that is not a polynomial.


In [ ]:
def my_poisson_1d(N, rho_over_eps0_interior, V_left, V_right):
    """Return V at the N interior points of [0, 1]."""
    h = 1.0 / (N + 1)
    A, b = poisson_1d_matrix(N, h, 0.0, V_left, V_right)           # replace from here with your own assembly
    b = b - h**2 * rho_over_eps0_interior
    return np.linalg.solve(A, b)


errors = []
for N in (7, 15, 31, 63):
    xi = np.arange(1, N + 1) / (N + 1)
    V = my_poisson_1d(N, np.pi**2 * np.sin(np.pi * xi), 0.0, 0.0)
    errors.append(np.abs(V - np.sin(np.pi * xi)).max())
    print(f"N = {N:3d}: max error {errors[-1]:.2e}")
ratios = np.array(errors[:-1]) / np.array(errors[1:])
print("error ratio per halving of h:", ratios, " (4 means second order)")
assert np.all(ratios > 3.5)

## 4. Verify

A 2D Poisson problem with a known answer. $V = \sin(\pi x)\sin(\pi y)$ vanishes on the boundary of the unit square and satisfies $\nabla^2V = -2\pi^2V$. So with $\rho/\epsilon_0 = 2\pi^2\sin(\pi x)\sin(\pi y)$ and $V = 0$ on the boundary, the solver must return $\sin(\pi x)\sin(\pi y)$, with an error that falls as $h^2$. This is the standard test for any Poisson solver: if the convergence rate is not two, something in the stencil or the boundary handling is wrong.


In [ ]:
def sparse_2d_poisson(n, rho_over_eps0):
    h = 1.0 / (n - 1); idx = np.arange(n * n).reshape(n, n)
    rows, cols, vals = [], [], []; b = np.zeros(n * n)
    for i in range(n):
        for j in range(n):
            k = idx[i, j]
            if i in (0, n - 1) or j in (0, n - 1):
                rows.append(k); cols.append(k); vals.append(1.0)
            else:
                rows.append(k); cols.append(k); vals.append(-4.0); b[k] = -h**2 * rho_over_eps0[i, j]
                for ni, nj in ((i - 1, j), (i + 1, j), (i, j - 1), (i, j + 1)):
                    rows.append(k); cols.append(idx[ni, nj]); vals.append(1.0)
    return spla.spsolve(sps.csr_matrix((vals, (rows, cols)), shape=(n * n, n * n)), b).reshape(n, n)

errs = []
for n in (11, 21, 41, 81):
    x = np.linspace(0, 1, n); X, Y = np.meshgrid(x, x, indexing="ij")
    exact = np.sin(np.pi * X) * np.sin(np.pi * Y)
    V = sparse_2d_poisson(n, 2 * np.pi**2 * exact)
    errs.append(np.abs(V - exact).max()); print(f"n = {n:2d}: max error {errs[-1]:.2e}")
print("error ratio per halving of h:", np.array(errs[:-1]) / np.array(errs[1:]), " (4 means second order)")
assert np.all(np.array(errs[:-1]) / np.array(errs[1:]) > 3.5)

## 5. Explore and rebuild

Explore. Jacobi on the $n\times n$ box needs of order $n^2$ sweeps. Measure the sweep count for $n = 11, 21, 41, 81$ and confirm the exponent. Then implement over-relaxation: replace the Gauss-Seidel update $V_i \leftarrow V_i^{\text{GS}}$ by $V_i \leftarrow V_i + \omega(V_i^{\text{GS}} - V_i)$ with $\omega$ between 1 and 2, and find the $\omega$ that minimises the sweep count for $n = 41$. Compare with the formula of section 3.4 and with `emlib.relax.sor_omega`, and confirm that the count now grows like $n$ rather than $n^2$. The gain is the reason notebook 02 reaches for SOR.

Rebuild. Close this notebook. On paper, derive the three-point formula from the Taylor expansion, assemble the $3\times3$ matrix of worked example 4 and solve it, do the three Jacobi sweeps of worked example 6, and show that a Jacobi sweep multiplies the mode $\sin(k\pi x)$ by $\cos(k\pi h)$. Then write the uniqueness argument in four sentences. In a blank editor, write the 1D tridiagonal assembly and the 2D sparse assembly from scratch and reproduce the second-order convergence tables.


## 6. Key results

| result | statement | holds when |
|---|---|---|
| Poisson's equation | $\nabla^2V = -\rho/\epsilon_0$ | always in electrostatics; $\nabla^2V = 0$ (Laplace) where $\rho = 0$ |
| boundary conditions | Dirichlet: $V$ given on the boundary. Neumann: $\partial V/\partial n$ given | one condition on every piece of the boundary gives one solution |
| plates | $V = V_0x/L$, $E = -V_0/L$; with uniform $\rho_0$: add $-\rho_0x^2/2\epsilon_0 + \rho_0Lx/2\epsilon_0$ | 1D, $V(0) = 0$, $V(L) = V_0$ |
| concentric spheres | $V = A/r + B$, $A = V_0ab/(b-a)$, $B = -V_0a/(b-a)$, $E_r = A/r^2$ | Laplace between radii $a$ (at $V_0$) and $b$ (at 0) |
| coaxial cylinders | $V = A\ln r + B$, $A = -V_0/\ln(b/a)$, $B = -A\ln b$, $E_r = -A/r$ | Laplace between radii $a$ (at $V_0$) and $b$ (at 0) |
| three-point formula | $V_i'' \approx (V_{i-1} - 2V_i + V_{i+1})/h^2$, error $-h^2V''''/12$ | exact for polynomials of degree $\le 3$ |
| five-point stencil | $V_{i\pm1,j} + V_{i,j\pm1} - 4V_{i,j} = -h^2\rho_{i,j}/\epsilon_0$ | 2D uniform grid |
| Jacobi update | $V_i \leftarrow \tfrac12(V_{i-1} + V_{i+1}) + h^2\rho_i/2\epsilon_0$ (1D), quarter of four neighbours (2D) | old values on the right; Gauss-Seidel uses new ones as they appear |
| mode damping | a Jacobi sweep multiplies $\sin(k\pi x)$ by $\cos(k\pi h)$ | 1D Laplace with zero boundary error |
| sweep counts | Jacobi and Gauss-Seidel: order $N^2$; SOR with $\omega_{\text{opt}} = 2/(1 + \sqrt{1 - \rho_J^2})$, $\rho_J = \cos(\pi h)$: order $N$ | $N$ points per side |
| uniqueness | two solutions with the same boundary data are identical | Dirichlet; Neumann fixes $V$ up to a constant |
| maximum principle | a solution of Laplace's equation has no interior maximum or minimum | charge-free region |
| mean-value property | the average of $V$ over a sphere or circle equals $V$ at its centre | Laplace's equation inside the sphere |
| Earnshaw | no stable electrostatic equilibrium for a free charge | charge-free region |
| ghost point | $V_{-1} = V_1 - 2hg$, boundary row $2V_1 - 2V_0 = -h^2\rho_0/\epsilon_0 + 2hg$ | Neumann $V'(x_0) = g$; symmetry plane is $g = 0$ |
| field from $V$ | $\mathbf{E} = -\nabla V$ by central differences; at a conductor face, one-sided $E_n = -(V_{\text{out}} - V_{\text{surf}})/h$ | interior nodes second order, surface first order |
| surface charge | $\sigma = \epsilon_0E_n$ | $E_n$ along the outward normal, just outside a conductor |
| total charge | $Q = \epsilon_0\oint\mathbf{E}\cdot d\mathbf{S}$ over any box enclosing the conductor | Gauss' law; `fields.box_flux` on the grid |


## 7. Warm-up answers

1. Substituting $\mathbf{E} = -\nabla V$ into $\nabla\cdot\mathbf{E} = \rho/\epsilon_0$ gives $\nabla^2V = -\rho/\epsilon_0$, Poisson's equation. With $\rho = 0$ it is Laplace's equation.
2. A differential equation inside a region together with conditions on the whole boundary of the region. A second-order equation needs either the value of $V$ (Dirichlet) or its normal derivative (Neumann) on every piece of the boundary; without that, infinitely many functions satisfy the equation.
3. $(V_{i-1} - 2V_i + V_{i+1})/h^2$. The error is $-h^2V''''/12$ plus higher powers of $h$: second order, and zero for polynomials up to degree three.
4. $V_i$ equals the average of its neighbours: $\tfrac12(V_{i-1} + V_{i+1})$ in 1D, a quarter of the sum of the four neighbours in 2D.
5. An interior maximum would be larger than all its neighbours, but the equation says it equals their average. So the extreme values sit on the boundary. Applied to the difference of two solutions this gives uniqueness.
6. About $N^2$ (times a constant that depends on the tolerance). Each sweep multiplies the slowest error mode by $\cos(\pi h) \approx 1 - \pi^2/2N^2$, so of order $N^2$ sweeps are needed to shrink it by a fixed factor. Over-relaxation brings this down to order $N$.
7. $\mathbf{E} = -\nabla V$ by central differences on the grid. The charge on a conductor is $\epsilon_0$ times the outward flux of $\mathbf{E}$ through a box enclosing it, or equivalently the sum over its surface of $\sigma = \epsilon_0E_n$ with $E_n$ from a one-sided difference at the surface.
